In [1]:
import torch
import numpy as np

from botorch.models.gp_regression import SingleTaskGP
from botorch.models.deterministic import GenericDeterministicModel
from botorch.models.transforms.input import Normalize
from botorch.models.transforms.outcome import Standardize
from botorch.models.model import ModelList

from gpytorch.mlls import ExactMarginalLogLikelihood

from botorch import fit_gpytorch_mll
from botorch.optim.optimize import optimize_acqf_discrete_local_search
from botorch.acquisition.multi_objective.logei import qLogNoisyExpectedHypervolumeImprovement
from botorch.utils.multi_objective.box_decompositions.dominated import DominatedPartitioning

import time
import warnings
from botorch.exceptions.warnings import NumericsWarning
warnings.filterwarnings("ignore", category=NumericsWarning)
warnings.filterwarnings("ignore", category = RuntimeWarning)

tkwargs = {
    "dtype": torch.double,
    "device": torch.device("cuda:3" if torch.cuda.is_available() else "cpu"),
}

# Define default parameters for the BO loop
REFERENCE = torch.tensor([-0.1, 0.0], **tkwargs)
BATCH_SIZE = 10
N_ITER = 10
MAX_CONC = 8
STEP = 0.5

# Define default hyperparams for the acqf optimization routine
DEFAULT_PARAMS = {
    "num_restarts": 10,
    "raw_samples": 2048,
    "max_batch_size": 2048,
    "max_tries": 100
}

In [2]:
from mlp import MLP

# Load the surrogate model
mdl = MLP().to(**tkwargs)
mdl.load_state_dict(torch.load('models/surrogate.pth', weights_only=True))
mdl.eval()

# Define a function to query the surrogate model
def surrogate_model(X):
    return mdl(X).detach()

# Define a function to query the deterministic model
def deterministic_model(X):
    return X.sum(dim=-1).unsqueeze(-1).detach()

In [3]:
def loadData():
    data = np.concatenate([
        np.load('data/data_initial.npy'),
        np.load('data/data_iter1.npy'),
        np.load('data/data_iter2.npy')
    ])
    return torch.Tensor(data).to(**tkwargs)

In [5]:
def initialize_model(train_X, train_Y):
    # Define deterministic model (on total concentration, doesnt require train_Y though)
    deterministic_model = GenericDeterministicModel(lambda x: x.sum(dim=-1, keepdim=True))
    # Define GP model (on viability)
    gp_model = SingleTaskGP(
        train_X,
        train_Y[:,1].unsqueeze(-1),
        input_transform = Normalize(train_X.shape[-1]),
        outcome_transform = Standardize(m=1)
    )
    model = ModelList(deterministic_model, gp_model)
    mll = ExactMarginalLogLikelihood(model.models[1].likelihood, model.models[1])
    return mll, model

def step_mobo(model, train_X, num_restarts, raw_samples, max_batch_size, max_tries):
    acq = qLogNoisyExpectedHypervolumeImprovement(
        model = model,
        ref_point = REFERENCE,
        X_baseline = train_X,
        prune_baseline = True
    )
    candidates, _ = optimize_acqf_discrete_local_search(
        acq_function = acq,
        q = BATCH_SIZE,
        discrete_choices = [torch.arange(0, MAX_CONC+STEP, STEP, **tkwargs)]*train_X.shape[-1],
        inequality_constraints = [
            (
                torch.arange(train_X.shape[-1], dtype = torch.int).to(tkwargs['device']),
                torch.full((train_X.shape[-1],), -1.0, **tkwargs),
                -MAX_CONC
            )
        ],
        num_restarts = num_restarts,
        raw_samples = raw_samples,
        max_batch_size = max_batch_size,
        max_tries = max_tries,
        unique = True
    )
    return candidates

def run_bo():
    # init_X = loadData()[:,:-2]
    # init_Y = loadData()[:,-2:]

    data = np.load('data/data_all.npy')
    data = torch.Tensor(data).to(**tkwargs)
    init_X = data[:,:-3]
    init_Y = data[:,-3:-1]

    mll, model = initialize_model(init_X, init_Y)

    train_X, train_Y = init_X.clone(), init_Y.clone()

    hvs = [DominatedPartitioning(REFERENCE, init_Y).compute_hypervolume().item()]
    times = [0.0]
    candidates_list = [init_X.cpu().tolist()]

    print(f"Iteration {0} | HV = {hvs[0]:.5f}, t = {times[0]}s")

    for i in range(1, N_ITER+1):
        start = time.time()
        fit_gpytorch_mll(mll)
        candidates = step_mobo(model, train_X, **DEFAULT_PARAMS)
        end = time.time()

        # Empty the cuda cache to save memory
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Compute new_X and new_Y
        new_X = candidates.detach()
        new_Y = torch.cat([deterministic_model(new_X), surrogate_model(new_X)], dim=-1)

        train_X = torch.cat([train_X, new_X], dim=0)
        train_Y = torch.cat([train_Y, new_Y], dim=0)

        # Compute the hypervolume
        hv = DominatedPartitioning(ref_point=REFERENCE, Y = train_Y).compute_hypervolume().item()

        # Save all relevant data to lists
        hvs.append(hv)
        times.append(end-start)
        candidates_list.append(candidates.cpu().tolist())

        mll, model = initialize_model(train_X, train_Y)

        print(f"Iteration {i} | HV = {hv:.5f}, t = {end-start:.2f}s")

    return hvs, times, candidates_list

In [6]:
import plotter

In [7]:
MAX_CONC = 6
N_ITER = 10

with torch.random.fork_rng(devices = [0]):
    hvs, times, candidates_list = run_bo()

plotter.generate_gif(candidates_list, MAX_CONC, filename = f'local-search-{MAX_CONC}M.gif')

Iteration 0 | HV = 600.61873, t = 0.0s


/home/dan/Research/cpa-bayesian-opt/py31/lib/python3.11/site-packages/torch/random.py:187: UserWarning: CUDA reports that you have 4 available devices, and you have used fork_rng without explicitly specifying which devices are being used. For safety, we initialize *every* CUDA device by default, which can be quite slow if you have a lot of CUDAs. If you know that you are only making use of a few CUDA devices, set the environment variable CUDA_VISIBLE_DEVICES or the 'devices' keyword argument of fork_rng with the set of devices you are actually using. For example, if you are using CPU only, set device.upper()_VISIBLE_DEVICES= or devices=[]; if you are using device 0 only, set CUDA_VISIBLE_DEVICES=0 or devices=[0].  To initialize all devices and suppress this warning, set the 'devices' keyword argument to `range(torch.cuda.device_count())`.
  warnings.warn(message)


Iteration 1 | HV = 605.30508, t = 12.03s
Iteration 2 | HV = 605.30508, t = 12.92s
Iteration 3 | HV = 605.30508, t = 12.10s
Iteration 4 | HV = 605.52435, t = 12.51s
Iteration 5 | HV = 605.52435, t = 13.36s
Iteration 6 | HV = 605.52435, t = 12.60s
Iteration 7 | HV = 605.52435, t = 12.91s
Iteration 8 | HV = 605.52435, t = 12.03s
Iteration 9 | HV = 605.52435, t = 14.85s
Iteration 10 | HV = 605.52435, t = 12.96s
GIF saved as local-search-6M.gif


In [7]:
MAX_CONC = 8
N_ITER = 10

with torch.random.fork_rng(devices = [0]):
    hvs, times, candidates_list = run_bo()

plotter.generate_gif(candidates_list, MAX_CONC, filename = f'local-search-{MAX_CONC}M.gif')

Iteration 0 | HV = 594.93836, t = 0.0s
Iteration 1 | HV = 693.96153, t = 6.63s
Iteration 2 | HV = 714.99090, t = 5.22s
Iteration 3 | HV = 721.31207, t = 5.42s
Iteration 4 | HV = 725.89922, t = 5.44s
Iteration 5 | HV = 729.92286, t = 5.22s
Iteration 6 | HV = 733.15002, t = 6.43s
Iteration 7 | HV = 734.18036, t = 5.82s
Iteration 8 | HV = 735.18199, t = 6.35s
Iteration 9 | HV = 736.50133, t = 6.15s
Iteration 10 | HV = 736.86051, t = 6.65s
GIF saved as local-search-8M.gif


In [8]:
MAX_CONC = 10
N_ITER = 10

with torch.random.fork_rng(devices = [0]):
    hvs, times, candidates_list = run_bo()

plotter.generate_gif(candidates_list, MAX_CONC, filename = f'local-search-{MAX_CONC}M.gif')

Iteration 0 | HV = 594.93836, t = 0.0s
Iteration 1 | HV = 698.64466, t = 7.08s
Iteration 2 | HV = 713.16615, t = 6.50s
Iteration 3 | HV = 781.90619, t = 6.71s
Iteration 4 | HV = 781.90619, t = 6.64s
Iteration 5 | HV = 788.70280, t = 6.22s
Iteration 6 | HV = 797.02340, t = 6.54s
Iteration 7 | HV = 812.31861, t = 6.07s
Iteration 8 | HV = 824.07529, t = 6.63s
Iteration 9 | HV = 833.74921, t = 5.66s
Iteration 10 | HV = 836.88761, t = 5.57s
GIF saved as local-search-10M.gif


In [20]:
MAX_CONC = 8
N_ITER = 10

BATCH_SIZE = 14

DEFAULT_PARAMS = {
    "num_restarts": 10,
    "raw_samples": 2048,
    "max_batch_size": 2048,
    "max_tries": 100
}

with torch.random.fork_rng(devices = [0]):
    hvs, times, candidates_list = run_bo()

plotter.generate_gif(candidates_list, MAX_CONC, filename = f'local-search-{MAX_CONC}M-q14.gif')

Iteration 0 | HV = 594.93836, t = 0.0s
Iteration 1 | HV = 700.74482, t = 13.36s
Iteration 2 | HV = 709.39311, t = 11.93s
Iteration 3 | HV = 721.34533, t = 11.75s
Iteration 4 | HV = 723.53884, t = 12.58s
Iteration 5 | HV = 726.05608, t = 12.91s
Iteration 6 | HV = 730.95491, t = 14.96s
Iteration 7 | HV = 737.97381, t = 14.75s
Iteration 8 | HV = 739.89328, t = 14.52s
Iteration 9 | HV = 739.89328, t = 14.68s
Iteration 10 | HV = 739.89328, t = 15.13s
GIF saved as local-search-8M-q14.gif
